# Embedding Model Baseline Evaluation

Evaluate baseline performance of **Qwen3-Embedding-0.6B** on Vietnamese medical text retrieval.

## Qwen3-Embedding Key Configuration (from Official Docs)
- **Context Length**: 32K tokens (8192 recommended for training)
- **Padding Side**: **LEFT** (CRITICAL - different from most models)
- **Pooling Strategy**: **last_token** (not mean pooling)
- **Instruction Format**: `Instruct: {task}\nQuery: {query}`
- **Recommended Tasks**: retrieval, classification, clustering
- **Model**: Qwen/Qwen3-Embedding-0.6B

Reference: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B

## 1. Setup

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from tqdm import tqdm
import wandb
import os

# Configuration (following Qwen3-Embedding official guidelines)
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 512  # For inference (can handle up to 32K)
BATCH_SIZE = 32

# Task instruction for medical retrieval
RETRIEVAL_TASK = "Tìm kiếm thông tin y tế liên quan"

# W&B project
WANDB_PROJECT = "vietnamese-med-rag-embedding"
WANDB_RUN = "baseline-qwen3-embedding-0.6B"

print(f"Using device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_LENGTH}")
print(f"Retrieval task: {RETRIEVAL_TASK}")

## 2. Initialize W&B

In [ ]:
# Initialize wandb
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "retrieval_task": RETRIEVAL_TASK,
        "padding_side": "left",  # Qwen3-Embedding requirement
        "pooling_strategy": "last_token"  # Qwen3-Embedding requirement
    }
)

print("W&B initialized successfully!")

## 3. Load Model and Tokenizer

In [ ]:
# Load tokenizer with LEFT padding (CRITICAL for Qwen3-Embedding)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="left"  # CRITICAL: Qwen3-Embedding requires LEFT padding
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.float16
).to(DEVICE)

model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Tokenizer padding side: {tokenizer.padding_side}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 4. Load Dataset

In [ ]:
# Load Vietnamese medical QA dataset
dataset = load_dataset("mtue29/vietnamese-medical-dataset")

# Split dataset if no test split exists
if "test" not in dataset:
    print("No test split found. Creating 80/10/10 split...")
    train_val = dataset["train"].train_test_split(test_size=0.2, seed=42)
    val_test = train_val["test"].train_test_split(test_size=0.5, seed=42)
    
    train_dataset = train_val["train"]
    val_dataset = val_test["train"]
    test_dataset = val_test["test"]
else:
    train_dataset = dataset["train"]
    val_dataset = dataset.get("validation", dataset["train"].train_test_split(test_size=0.1, seed=42)["test"])
    test_dataset = dataset["test"]

# Limit test set size for faster evaluation
test_dataset = test_dataset.select(range(min(500, len(test_dataset))))

print(f"Test dataset size: {len(test_dataset)}")
print(f"\nSample:")
print(f"Question: {test_dataset[0]['question']}")
print(f"Answer: {test_dataset[0]['answer'][:100]}...")

## 5. Define Embedding Functions

In [ ]:
def format_instruction(text: str, is_query: bool = True) -> str:
    """Format text with Qwen3-Embedding instruction template.
    
    Format: Instruct: {task}\nQuery: {query}
    Reference: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
    """
    if is_query:
        return f"Instruct: {RETRIEVAL_TASK}\nQuery: {text}"
    else:
        return text  # Documents don't need instruction prefix


def encode_texts(texts: list, is_query: bool = True, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """Encode texts to embeddings using Qwen3-Embedding.
    
    CRITICAL:
    - Uses LEFT padding (tokenizer.padding_side="left")
    - Uses last_token pooling (not mean pooling)
    - Applies instruction format for queries
    """
    all_embeddings = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch_texts = texts[i:i+batch_size]
        
        # Format with instruction if query
        if is_query:
            batch_texts = [format_instruction(t, is_query=True) for t in batch_texts]
        
        # Tokenize with LEFT padding
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            
            # Last token pooling (CRITICAL for Qwen3-Embedding)
            # Since we use LEFT padding, last token is at position -1
            embeddings = outputs.last_hidden_state[:, -1, :]  # [batch_size, hidden_dim]
        
        all_embeddings.append(embeddings.cpu().numpy())
    
    # Concatenate all batches
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    
    # Normalize embeddings
    all_embeddings = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    
    return all_embeddings


print("Embedding functions defined.")
print("Using LEFT padding + last_token pooling (Qwen3-Embedding requirement)")

## 6. Encode Questions and Answers

In [ ]:
# Extract questions and answers
questions = [sample["question"] for sample in test_dataset]
answers = [sample["answer"] for sample in test_dataset]

# Encode questions (as queries with instruction)
print("Encoding questions (as queries)...")
question_embeddings = encode_texts(questions, is_query=True)

# Encode answers (as documents without instruction)
print("\nEncoding answers (as documents)...")
answer_embeddings = encode_texts(answers, is_query=False)

print(f"\nQuestion embeddings shape: {question_embeddings.shape}")
print(f"Answer embeddings shape: {answer_embeddings.shape}")

## 7. Calculate Retrieval Metrics

In [ ]:
# Calculate similarity matrix (question vs all answers)
similarity_matrix = cosine_similarity(question_embeddings, answer_embeddings)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

# Calculate retrieval metrics
def calculate_retrieval_metrics(similarity_matrix):
    """
    Calculate retrieval metrics.
    
    For each question, the correct answer is at the same index.
    We check if the most similar answer is the correct one.
    """
    n_samples = similarity_matrix.shape[0]
    
    # Top-k accuracy
    top_k_values = [1, 3, 5, 10]
    top_k_accuracy = {}
    
    for k in top_k_values:
        correct = 0
        for i in range(n_samples):
            # Get top-k indices
            top_k_indices = np.argsort(similarity_matrix[i])[::-1][:k]
            # Check if correct answer (index i) is in top-k
            if i in top_k_indices:
                correct += 1
        
        top_k_accuracy[f"top_{k}_accuracy"] = correct / n_samples
    
    # Mean Reciprocal Rank (MRR)
    mrr_sum = 0
    for i in range(n_samples):
        # Get rank of correct answer
        sorted_indices = np.argsort(similarity_matrix[i])[::-1]
        rank = np.where(sorted_indices == i)[0][0] + 1  # 1-indexed
        mrr_sum += 1.0 / rank
    
    mrr = mrr_sum / n_samples
    
    # NDCG@10
    ndcg_scores = []
    for i in range(n_samples):
        # Create ground truth (only correct answer has relevance 1)
        true_relevance = np.zeros(n_samples)
        true_relevance[i] = 1
        
        # Predicted scores
        pred_scores = similarity_matrix[i]
        
        # Calculate NDCG@10
        ndcg = ndcg_score([true_relevance], [pred_scores], k=10)
        ndcg_scores.append(ndcg)
    
    mean_ndcg_10 = np.mean(ndcg_scores)
    
    return {
        **top_k_accuracy,
        "mrr": mrr,
        "ndcg@10": mean_ndcg_10
    }


# Calculate metrics
metrics = calculate_retrieval_metrics(similarity_matrix)

print("\n=== Retrieval Metrics (Baseline Qwen3-Embedding-0.6B) ===")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")
    wandb.log({f"eval/{metric}": value})

## 8. Analyze Results

In [ ]:
# Show examples of correct and incorrect retrievals
print("\n=== Sample Retrievals ===")

for i in range(min(3, len(test_dataset))):
    question = questions[i]
    correct_answer = answers[i]
    
    # Get top-3 most similar answers
    top_3_indices = np.argsort(similarity_matrix[i])[::-1][:3]
    
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {question[:100]}...")
    print(f"Correct Answer (rank={np.where(np.argsort(similarity_matrix[i])[::-1] == i)[0][0] + 1}): {correct_answer[:80]}...")
    
    print("\nTop-3 Retrieved Answers:")
    for rank, idx in enumerate(top_3_indices, 1):
        similarity = similarity_matrix[i][idx]
        is_correct = "✓ CORRECT" if idx == i else "✗ Wrong"
        print(f"{rank}. [Sim: {similarity:.3f}] {is_correct}")
        print(f"   {answers[idx][:80]}...")

print("\n" + "="*60)

## 9. Save Results

In [ ]:
# Save embeddings for later use
output_dir = "../outputs/baseline_embeddings"
os.makedirs(output_dir, exist_ok=True)

np.save(f"{output_dir}/question_embeddings.npy", question_embeddings)
np.save(f"{output_dir}/answer_embeddings.npy", answer_embeddings)
np.save(f"{output_dir}/similarity_matrix.npy", similarity_matrix)

print(f"Embeddings saved to {output_dir}")

# Save metrics to W&B
wandb.log({"final_metrics": metrics})

# Finish W&B run
wandb.finish()

print("\nBaseline evaluation completed!")
print(f"Results logged to W&B project: {WANDB_PROJECT}")